# Chapter 3 Practical 06: Cold Start, Sparsity, Clustering, and Temporal Dynamics

Learning objectives:
- Detect cold-start users and items.
- Apply minimum-overlap and shrinkage.
- Cluster users to reduce neighbor search.
- Add time-decay weights to recent interactions.

Slide connection: limits of memory-based CF, practical mitigations, clustering for scalability, and temporal dynamics.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


In [ ]:
user_counts = ratings_named.groupby("user_id").size().rename("ratings_count")
item_counts = ratings_named.groupby("title").size().rename("ratings_count")

print("Cold-start-like users:")
display(user_counts[user_counts <= 2])
print("Cold-start-like items:")
display(item_counts[item_counts <= 2])


In [ ]:
def pearson_with_shrinkage(matrix, user_a, user_b, min_overlap=2, alpha=3):
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    overlap = pair.shape[1]
    if overlap < min_overlap:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    raw = np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1]
    return raw * overlap / (overlap + alpha)

rows = []
for other in rating_matrix.index.drop("Karen"):
    rows.append({
        "neighbor": other,
        "shrunk_similarity": pearson_with_shrinkage(rating_matrix, "Karen", other),
    })
pd.DataFrame(rows).sort_values("shrunk_similarity", ascending=False).round(3)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

filled = rating_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)
scaled = StandardScaler().fit_transform(filled)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = pd.Series(kmeans.fit_predict(scaled), index=rating_matrix.index, name="cluster")
clusters.to_frame().sort_values("cluster")


In [ ]:
target_user = "Karen"
target_cluster = clusters.loc[target_user]
candidate_neighbors = clusters[clusters.eq(target_cluster)].index.drop(target_user)
print(f"Compare Karen only with users in cluster {target_cluster}: {candidate_neighbors.tolist()}")


In [ ]:
decay_lambda = 0.01
ratings_named["time_weight"] = np.exp(-decay_lambda * ratings_named["days_ago"])
ratings_named["weighted_rating"] = ratings_named["rating"] * ratings_named["time_weight"]

ratings_named[["user_id", "title", "rating", "days_ago", "time_weight", "weighted_rating"]].sort_values("days_ago").head(10).round(3)


In [ ]:
recent_profile = (
    ratings_named.groupby("title")
    .apply(lambda g: np.average(g["rating"], weights=g["time_weight"]))
    .rename("time_weighted_mean")
    .sort_values(ascending=False)
)
recent_profile.head(5).round(2)


Exercises:
1. Increase `decay_lambda`. Which older ratings fade fastest?
2. Change the number of clusters and compare neighbor candidates.
